# PII Model Fine-Tuning Pipeline (Enhanced)

This notebook contains the complete end-to-end pipeline for:
1. **Enhanced Dataset Generation**: Extracts real Synthea FHIR data and generates training records with hard-negative examples.
2. **Data Processing & Subword Alignment**: Splits dataset, tokenizes with RoBERTa, and aligns word labels to subwords.
3. **Model Fine-Tuning**: Trains the model using Hugging Face's `Trainer` (5 epochs).
4. **Evaluation**: Evaluates the fine-tuned model and prints TP, FP, FN, Precision, Recall, and F1-Score for each entity type.
5. **PII Inference**: Runs prediction on a sample EHR record with regression checks for known FP patterns.

### Key Improvements Over Original Pipeline
- **New entity types**: US_PASSPORT, RACE, NRP (previously had zero training examples)
- **Hard-negative examples**: MRN UUIDs tagged as O (not US_DRIVER_LICENSE), language names as O (not RACE)
- **Race+gender adjacency**: Teaches model to distinguish race from gender in adjacent context
- **PII only (not PHI)**: Medical identifiers (MRN, NPI, Medical License) are explicitly non-PII

## 1. Enhanced Dataset Generation

This step runs two scripts:
1. `synthea_extractor.py` — Extracts demographics from 115 Synthea FHIR bundles
2. `build_enhanced_dataset.py` — Generates 2500+ NER records with hard negatives

If `enhanced_dataset.json` already exists, this cell is optional.

In [1]:
import os
import sys
import subprocess

# Find repository root robustly
current_dir = os.getcwd()
while current_dir and not os.path.exists(os.path.join(current_dir, "models")):
    parent = os.path.dirname(current_dir)
    if parent == current_dir:
        break
    current_dir = parent
REPO_ROOT = current_dir
TRAINING_DIR = os.path.join(REPO_ROOT, "training")

# Step 1: Extract Synthea FHIR data
extractor = os.path.join(TRAINING_DIR, 'synthea_extractor.py')
if os.path.exists(extractor):
    print('Running Synthea extractor...')
    subprocess.run([sys.executable, extractor], check=True)
else:
    print(f'Extractor not found at {extractor}, skipping.')

# Step 2: Build enhanced dataset
builder = os.path.join(TRAINING_DIR, 'build_enhanced_dataset.py')
if os.path.exists(builder):
    print('\nBuilding enhanced dataset...')
    subprocess.run([sys.executable, builder], check=True)
else:
    print(f'Builder not found at {builder}, skipping.')

# Verify output
import json
data_path = os.path.join(TRAINING_DIR, 'data', 'enhanced_dataset.json')
with open(data_path) as f:
    dataset_check = json.load(f)
print(f'\nEnhanced dataset: {len(dataset_check)} records ready for training.')

Running Synthea extractor...


Processing Synthea FHIR files from: /Users/deepak/projects/FineTune/synthea_sample_data_fhir_latest

Extracted 115 patient records to /Users/deepak/projects/FineTune/training/data/synthea_extracted.json

Field coverage:
  has_dl: 97/115
  has_passport: 94/115
  has_phone: 115/115
  has_race: 115/115
  has_ssn: 115/115

Sample patient:
  Name: Ms. Afton574 Casper496
  Gender: female
  Race: White
  SSN: 999-91-1291
  Passport: X35482474X
  DL: S99944180
  Address: 183 Hagenes Stravenue Suite 78, Plymouth, MA, 02360
  Practitioners: ['Dr. Cecil300 Fahey393', 'Dr. Mica978 Lemke654', 'Dr. Gabriel934 Reilly981']
  Organizations: ['LIFE CARE CENTER OF PLYMOUTH', 'Fitchburg Outpatient Clinic', 'SOUTH SHORE PRIMARY AND URGENT CARE LLC']
  Conditions: ['Lack of access to transportation (finding)', 'Prediabetes (finding)', 'Anemia (disorder)']

Building enhanced dataset...
Loaded 115 Synthea patients
Loaded 1000 existing synthetic records
Generated 805 Synthea-based records
Generated 713 hard-ne

## 2. Dataset Processing and Token Alignment

In [2]:
import os
import json
import torch
import numpy as np
import pandas as pd
from datasets import Dataset
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForTokenClassification

# Disable MPS high watermark memory allocation limits to prevent OOM errors
os.environ["PYTORCH_MPS_HIGH_WATERMARK_RATIO"] = "0.0"

# Find repository root robustly
current_dir = os.getcwd()
while current_dir and not os.path.exists(os.path.join(current_dir, "models")):
    parent = os.path.dirname(current_dir)
    if parent == current_dir:
        break
    current_dir = parent
BASE_DIR = current_dir

# Load enhanced dataset robustly
data_path = os.path.join(BASE_DIR, "training", "data", "enhanced_dataset.json")
print(f"Loading dataset from {data_path}...")
with open(data_path, "r", encoding="utf-8") as f:
    dataset = json.load(f)

# Split into Train (80%) and Validation (20%)
train_data, val_data = train_test_split(dataset, test_size=0.2, random_state=42)
print(f"Train records: {len(train_data)}, Validation records: {len(val_data)}")

# Set paths
MODEL_PATH = os.path.join(BASE_DIR, "models", "roberta-pii-synthetic")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print(f"Using device: {DEVICE}")

# Load model & tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    "FacebookAI/roberta-base",
    add_prefix_space=True,
    max_length=512
)
model = AutoModelForTokenClassification.from_pretrained(MODEL_PATH)
model.to(DEVICE)

id2label = model.config.id2label
label2id = model.config.label2id
label_list = [id2label[i] for i in range(len(id2label))]
print(f"Loaded pretrained model labels. Count: {len(label_list)}")

# Prepare Datasets
train_df = pd.DataFrame(train_data)
val_df = pd.DataFrame(val_data)

train_df['ner_tags_ids'] = train_df['ner_tags'].apply(lambda tags: [label2id[t] for t in tags])
val_df['ner_tags_ids'] = val_df['ner_tags'].apply(lambda tags: [label2id[t] for t in tags])

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

def align_labels_with_tokens(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        max_length=512
    )
    
    labels = []
    for i, label_ids in enumerate(examples["ner_tags_ids"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids_aligned = []
        
        for word_idx in word_ids:
            if word_idx is None:
                label_ids_aligned.append(-100)
            elif word_idx != previous_word_idx:
                label_ids_aligned.append(label_ids[word_idx])
            else:
                label_ids_aligned.append(-100)
            previous_word_idx = word_idx
            
        labels.append(label_ids_aligned)
        
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

tokenized_train = train_dataset.map(align_labels_with_tokens, batched=True, remove_columns=train_dataset.column_names)
tokenized_val = val_dataset.map(align_labels_with_tokens, batched=True, remove_columns=val_dataset.column_names)

Loading dataset from /Users/deepak/projects/FineTune/training/data/enhanced_dataset.json...
Train records: 2014, Validation records: 504
Using device: mps


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loaded pretrained model labels. Count: 51


Map:   0%|          | 0/2014 [00:00<?, ? examples/s]

Map:   0%|          | 0/504 [00:00<?, ? examples/s]

## 3. Model Fine-Tuning

Training for **5 epochs** (increased from 3) to ensure convergence on the expanded dataset with new entity types.
To reduce memory usage and prevent out-of-memory errors on Apple Silicon GPUs (MPS), `per_device_train_batch_size` has been set to 4 with `gradient_accumulation_steps=2` (resulting in the same effective batch size of 8).

In [3]:
from transformers import DataCollatorForTokenClassification, TrainingArguments, Trainer
from seqeval.metrics import precision_score, recall_score, f1_score

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)
    
    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    
    return {
        "precision": precision_score(true_labels, true_predictions),
        "recall": recall_score(true_labels, true_predictions),
        "f1": f1_score(true_labels, true_predictions),
    }

data_collator = DataCollatorForTokenClassification(tokenizer)

training_args = TrainingArguments(
    output_dir=os.path.join(BASE_DIR, "training", "artifacts", "results"),
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_steps=20,
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

print("Starting fine-tuning...")
trainer.train()

# Save final weights
OUTPUT_MODEL_DIR = os.path.join(BASE_DIR, "models", "roberta-pii-synthetic")
trainer.save_model(OUTPUT_MODEL_DIR)
tokenizer.save_pretrained(OUTPUT_MODEL_DIR)
print(f"Fine-tuning complete. Model saved to {OUTPUT_MODEL_DIR}")

Starting fine-tuning...


/opt/homebrew/lib/python3.14/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.003639,0.000181,0.999662,0.999662,0.999662
2,0.000448,0.000062,1.000000,1.000000,1.000000
3,0.000289,0.000044,1.000000,1.000000,1.000000
4,0.000217,0.000040,1.000000,1.000000,1.000000
5,0.000181,0.000034,1.000000,1.000000,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/opt/homebrew/lib/python3.14/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/opt/homebrew/lib/python3.14/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-tuning complete. Model saved to /Users/deepak/projects/FineTune/models/roberta-pii-synthetic


## 4. Rigorous Evaluation (TP, FP, FN, Precision, Recall, F1)

This block runs predictions on the validation set, counts exact entity True Positives, False Positives, and False Negatives, and prints them in a summary table.

In [4]:
from seqeval.metrics.sequence_labeling import get_entities

# Load the best fine-tuned model for evaluation
eval_model = AutoModelForTokenClassification.from_pretrained(OUTPUT_MODEL_DIR).to(DEVICE)
eval_model.eval()

all_true = []
all_pred = []

for item in val_data:
    tokens = item["tokens"]
    true_tags = item["ner_tags"]
    
    enc = tokenizer(
        tokens,
        is_split_into_words=True,
        truncation=True,
        max_length=512,
        return_tensors="pt"
    ).to(DEVICE)
    
    with torch.no_grad():
        logits = eval_model(**enc).logits
    
    preds = torch.argmax(logits, dim=2)[0].cpu().tolist()
    word_ids = enc.word_ids(batch_index=0)
    
    pred_tags = []
    seen_words = set()
    for subword_idx, word_idx in enumerate(word_ids):
        if word_idx is None or word_idx in seen_words:
            continue
        seen_words.add(word_idx)
        pred_tags.append(id2label[preds[subword_idx]])
        
    if len(pred_tags) == len(true_tags):
        all_true.append(true_tags)
        all_pred.append(pred_tags)

counts = {}
for true_sent, pred_sent in zip(all_true, all_pred):
    true_entities = set(get_entities(true_sent))
    pred_entities = set(get_entities(pred_sent))
    
    tps = true_entities.intersection(pred_entities)
    fps = pred_entities - true_entities
    fns = true_entities - pred_entities
    
    for ent_type, _, _ in tps:
        counts.setdefault(ent_type, {'tp': 0, 'fp': 0, 'fn': 0})['tp'] += 1
    for ent_type, _, _ in fps:
        counts.setdefault(ent_type, {'tp': 0, 'fp': 0, 'fn': 0})['fp'] += 1
    for ent_type, _, _ in fns:
        counts.setdefault(ent_type, {'tp': 0, 'fp': 0, 'fn': 0})['fn'] += 1

report_rows = []
for ent_type, metrics in sorted(counts.items()):
    tp, fp, fn = metrics['tp'], metrics['fp'], metrics['fn']
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    report_rows.append({
        "Entity": ent_type,
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "Precision": precision,
        "Recall": recall,
        "F1-Score": f1
    })

df_report = pd.DataFrame(report_rows)
print("\n==================================================================")
print("EVALUATION METRICS TABLE (VALIDATION SET)")
print("==================================================================")
print(df_report.to_string(index=False))
print("==================================================================")

total_fps = df_report["FP"].sum()
total_fns = df_report["FN"].sum()
print(f"Total False Positives: {total_fps}")
print(f"Total False Negatives: {total_fns}")
if total_fps == 0 and total_fns == 0:
    print("SUCCESS: Zero False Positives and Zero False Negatives achieved!")
else:
    print("WARNING: Non-zero FP or FN found. Refine training settings.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


EVALUATION METRICS TABLE (VALIDATION SET)
           Entity  TP  FP  FN  Precision  Recall  F1-Score
      CREDIT_CARD  49   0   0        1.0     1.0       1.0
             DATE 456   0   0        1.0     1.0       1.0
    EMAIL_ADDRESS  97   0   0        1.0     1.0       1.0
           GENDER 249   0   0        1.0     1.0       1.0
       IP_ADDRESS  61   0   0        1.0     1.0       1.0
         LOCATION 209   0   0        1.0     1.0       1.0
              NRP  23   0   0        1.0     1.0       1.0
     ORGANIZATION 220   0   0        1.0     1.0       1.0
           PERSON 553   0   0        1.0     1.0       1.0
     PHONE_NUMBER 205   0   0        1.0     1.0       1.0
             RACE 135   0   0        1.0     1.0       1.0
             TIME 121   0   0        1.0     1.0       1.0
              URL  70   0   0        1.0     1.0       1.0
   US_BANK_NUMBER  50   0   0        1.0     1.0       1.0
US_DRIVER_LICENSE 111   0   0        1.0     1.0       1.0
      US_PASS

## 5. PII Inference & Regression Test

Runs inference on the Afton574 Casper496 EHR record — the test case that originally exposed 4 False Positives:
- `"white"` wrongly tagged as GENDER (should be RACE)
- `"English"` wrongly tagged as RACE (should be O — it's a language)
- MRN UUID wrongly tagged as US_DRIVER_LICENSE (should be O — not PII)
- Passport wrongly tagged as US_SSN (should be US_PASSPORT)

In [5]:
import re

test_text = (
    "Afton574 Casper496 is a white female born on September 9, 1922, "
    "whose mother's maiden name is Leonia336 Gaylord332. She is never married, "
    "resides at 183 Hagenes Stravenue Suite 78 in Plymouth, Massachusetts "
    "(postal code 02360), and speaks English. Her medical record number is "
    "b7672a45-fe62-d8e3-9721-809e3ede1700, and she has a Social Security Number "
    "of 999-91-1291, a driver's license number of S99944180, and a passport "
    "number of X35482474X. She has received care primarily from Dr. Isaac321 "
    "Kub800 at Plymouth Carver Primary Care, P.C., and her insurance is provided "
    "by Medicaid, Medicare, and Dual Eligible programs. Her documented medical "
    "history includes lack of access to transportation, prediabetes, anemia, "
    "obesity, hypertriglyceridemia, neuropathy due to type 2 diabetes mellitus, "
    "sleep disorder, obstructive sleep apnea syndrome, and hyperglycemia"
)

# Tokenize and predict
words = re.findall(r"\w+|[^\w\s]", test_text)
enc = tokenizer(
    words,
    is_split_into_words=True,
    truncation=True,
    max_length=512,
    return_tensors="pt"
).to(DEVICE)

with torch.no_grad():
    logits = eval_model(**enc).logits

probs = torch.softmax(logits, dim=2)[0]
preds = torch.argmax(logits, dim=2)[0].cpu().tolist()
word_ids = enc.word_ids(batch_index=0)

pred_tags = []
pred_confs = []
seen_words = set()
for subword_idx, word_idx in enumerate(word_ids):
    if word_idx is None or word_idx in seen_words:
        continue
    seen_words.add(word_idx)
    pred_tags.append(id2label[preds[subword_idx]])
    pred_confs.append(probs[subword_idx, preds[subword_idx]].item())

if len(pred_tags) < len(words):
    pred_tags.extend(["O"] * (len(words) - len(pred_tags)))
    pred_confs.extend([1.0] * (len(words) - len(pred_confs)))
else:
    pred_tags = pred_tags[:len(words)]
    pred_confs = pred_confs[:len(words)]

# BIO Grouping
word_spans = []
current_idx = 0
for w in words:
    start_idx = test_text.find(w, current_idx)
    if start_idx == -1:
        start_idx = test_text.find(w)
    end_idx = start_idx + len(w)
    word_spans.append((start_idx, end_idx))
    current_idx = end_idx

entities = []
current_entity = None

for idx, (word, tag, conf) in enumerate(zip(words, pred_tags, pred_confs)):
    w_start, w_end = word_spans[idx]
    if w_start == -1:
        continue

    if tag.startswith("B-"):
        entity_type = tag.split("-", 1)[1]
        if current_entity:
            entities.append(current_entity)
        current_entity = {
            "entity": entity_type,
            "start": w_start,
            "end": w_end,
            "text": test_text[w_start:w_end],
            "confidences": [conf]
        }
    elif tag.startswith("I-"):
        entity_type = tag.split("-", 1)[1]
        if current_entity and entity_type == current_entity["entity"]:
            current_entity["end"] = w_end
            current_entity["text"] = test_text[current_entity["start"]:w_end]
            current_entity["confidences"].append(conf)
        else:
            if current_entity:
                entities.append(current_entity)
            current_entity = {
                "entity": entity_type,
                "start": w_start,
                "end": w_end,
                "text": test_text[w_start:w_end],
                "confidences": [conf]
            }
    else:
        if current_entity:
            entities.append(current_entity)
            current_entity = None

if current_entity:
    entities.append(current_entity)

# Format output
output_entities = []
for ent in entities:
    avg_conf = sum(ent["confidences"]) / len(ent["confidences"])
    output_entities.append({
        "text": ent["text"],
        "entity": ent["entity"],
        "confidence": round(avg_conf, 4)
    })

print("\nGrouped Predictions Output:")
print(json.dumps({"entities": output_entities}, indent=2))

# ── Regression Checks ──────────────────────────────────────────────
print("\n" + "=" * 60)
print("REGRESSION CHECKS (Known FP Patterns)")
print("=" * 60)

def has_entity(etype, substring):
    return any(e["entity"] == etype and substring in e["text"] for e in output_entities)

checks = [
    ("PERSON",            "Afton574 Casper496",   True,  "Patient name detected"),
    ("GENDER",            "female",               True,  "Gender detected"),
    ("RACE",              "white",                True,  "Race detected (not GENDER)"),
    ("DATE",              "September",            True,  "Birth date detected"),
    ("US_SSN",            "999-91-1291",          True,  "SSN detected"),
    ("US_DRIVER_LICENSE", "S99944180",            True,  "Driver's license detected"),
    ("US_PASSPORT",       "X35482474X",           True,  "Passport detected (not US_SSN)"),
    ("GENDER",            "white",                False, "white must NOT be GENDER"),
    ("RACE",              "English",              False, "English must NOT be RACE"),
    ("US_DRIVER_LICENSE", "b7672a45",             False, "MRN UUID must NOT be DL"),
    ("US_SSN",            "X35482474X",           False, "Passport must NOT be US_SSN"),
]

all_passed = True
for etype, substr, should_exist, msg in checks:
    found = has_entity(etype, substr)
    ok = (found == should_exist)
    status = "✅" if ok else "❌"
    if not ok:
        all_passed = False
    print(f"  {status} {msg}: {'PASS' if ok else 'FAIL'}")

print()
if all_passed:
    print("🎉 ALL REGRESSION CHECKS PASSED!")
else:
    print("⚠️  Some regression checks failed. Review the predictions above.")


Grouped Predictions Output:
{
  "entities": [
    {
      "text": "Afton574 Casper496",
      "entity": "PERSON",
      "confidence": 0.9999
    },
    {
      "text": "white",
      "entity": "RACE",
      "confidence": 0.9998
    },
    {
      "text": "female",
      "entity": "GENDER",
      "confidence": 0.9998
    },
    {
      "text": "September 9, 1922",
      "entity": "DATE",
      "confidence": 1.0
    },
    {
      "text": "Leonia336 Gaylord332",
      "entity": "PERSON",
      "confidence": 0.9999
    },
    {
      "text": "183 Hagenes Stravenue Suite 78 in Plymouth, Massachusetts",
      "entity": "LOCATION",
      "confidence": 0.9999
    },
    {
      "text": "02360",
      "entity": "LOCATION",
      "confidence": 0.9997
    },
    {
      "text": "999-91-1291",
      "entity": "US_SSN",
      "confidence": 0.9999
    },
    {
      "text": "S99944180",
      "entity": "US_DRIVER_LICENSE",
      "confidence": 0.9999
    },
    {
      "text": "X35482474X",
      "